# pypasi against the configuration of the method paper [6]

Reviewer 2, comment 5, asks whether pypasi is more than a repackaging of the code of [6]
(Kourkoumelis, *Chemometr. Intell. Lab. Syst.* 279 (2026) 105914). This notebook runs the
acquisition audit with **the configuration of [6]**, then changes one component at a time
until it reaches the pypasi default. Every number is read on the same data and splits.

| Component | [6] | pypasi default |
|---|---|---|
| Preprocessing | L2 vector normalization, then per-channel standardization | SNV |
| Band models | logistic regression | PLS-DA (10 components) |
| Evidence calibration | none | one temperature, fitted on held-out isolates |
| Topology | chain (neighbouring bands only) | complete graph |
| Conflict descriptor | eREDG | DG, chosen on the select isolates |

The regime (H1), the number of bands (7) and the band range (400–1792 cm⁻¹) are the same in
both. Defects are planted before preprocessing, as in the paper.

**Three outcomes**, all over five patient-disjoint splits:

- conflict against the planted defects;
- conflict against the real 2018 → 2019 instrument change;
- external accuracy.

**Requirements.** `pypasi >= 0.5.5` and the Ho et al. `.npy` files in `Bacteria/`. Runtime is about 15 minutes.

## 0 · Setup (shared with the other revision notebooks)

In [1]:
# --- shared setup: the data, the patient-disjoint split and the defects of the paper ---
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
import pypasi
from pypasi import BandNegotiationClassifier, BandSet, ControlProfile, PLSDA, Triage, degrade
from pypasi.datasets import load_bacteria

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Control limits built from a random split")
assert tuple(int(p) for p in pypasi.__version__.split(".")[:3]) >= (0, 5, 5), "needs pypasi >= 0.5.5"

DATA_DIR = Path("Bacteria")
SEED = 0
# the configuration of softwarex_examples.ipynb; every function below defaults to it
N_BANDS, TOPOLOGY, REGIME, N_COMPONENTS, SCALE_NAME = 7, "complete", "H1", 10, "SNV"
CALIBRATION, CAL_SPLIT = "auto", 0.25
REVIEW_RATE, NOVELTY_Q = 0.10, 0.99
K_SIGMA, LASER_NM, PROFILE_STATS = 3.0, 633.0, ("mean_stress", "band_confidence")
BATCH_N, LOCALISE_REPEATS = 100, 6
SPECTRA_PER_PATIENT = 400


def minmax(X):
    X = np.asarray(X, float)
    lo, hi = X.min(1, keepdims=True), X.max(1, keepdims=True)
    return (X - lo) / np.where(hi > lo, hi - lo, 1.0)


def snv(X):
    X = np.asarray(X, float)
    s = X.std(1, keepdims=True)
    return (X - X.mean(1, keepdims=True)) / np.where(s > 0, s, 1.0)


SCALERS = {"SNV": snv, "min-max": lambda X: np.asarray(X, float)}

data = load_bacteria(DATA_DIR, normalize=False)
AXIS = data.axis
MM_all, y_all = minmax(data.X_dev), data.y_dev      # 2018 cohort, per-spectrum min-max
MM_B, y_B = minmax(data.X_test), data.y_test        # 2019 cohort (external)

# Patient = block of 400 consecutive spectra; verified, never assumed
groups = np.arange(len(y_all)) // SPECTRA_PER_PATIENT
for g in np.unique(groups):
    assert np.unique(y_all[groups == g]).size == 1, "row order is not grouped by patient"
N_PAT = groups.max() + 1
PAT_CLASS = np.array([y_all[groups == g][0] for g in range(N_PAT)])

BAND_RANGE = (400.0, min(1800.0, float(AXIS.max())))
REF_BANDS = BandSet.equal_width(AXIS, 7, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
REF_EDGES = [(b.lo, b.hi) for b in REF_BANDS.bands]   # where local defects are planted


def _take(pool, per_class, rng):
    taken = []
    for c in np.unique(PAT_CLASS[pool]):
        taken.extend(rng.choice(pool[PAT_CLASS[pool] == c], per_class, replace=False))
    taken = np.array(sorted(taken))
    return taken, np.setdiff1d(pool, taken)


def make_split(split_repeat=0):
    """The paper's split (fit 10 | limits 5 | select 5 | qc 5 isolates) and its defects."""
    rng = np.random.default_rng(SEED + split_repeat)
    pool = np.arange(N_PAT)
    p_qc, pool = _take(pool, 1, rng)
    p_sel, pool = _take(pool, 1, rng)
    p_lim, pool = _take(pool, 1, rng)
    rows = lambda p: np.flatnonzero(np.isin(groups, p))  # noqa: E731
    ix = {"fit": rows(pool), "lim": rows(p_lim), "sel": rows(p_sel), "qc": rows(p_qc)}
    ix["ref"] = np.sort(np.concatenate([ix["lim"], ix["sel"], ix["qc"]]))
    return {
        "repeat": split_repeat, "p_qc": p_qc, "ix": ix,
        "chart_patients": np.sort(np.concatenate([p_lim, p_sel])),
        "sel_def": degrade.plant(MM_all[ix["sel"]], AXIS, edges=REF_EDGES, random_state=SEED + 77),
        "qc_def": degrade.plant(MM_all[ix["qc"]], AXIS, edges=REF_EDGES, random_state=SEED),
    }


def binf(X, f):
    """Average f adjacent channels: a coarser detector sampling."""
    X = np.asarray(X, float)
    if f == 1:
        return X
    n = (X.shape[-1] // f) * f
    return X[..., :n].reshape(*X.shape[:-1], n // f, f).mean(-1)


def fit_model(sp, *, n_bands=N_BANDS, topology=TOPOLOGY, scale=SCALE_NAME, mode="equal",
              regime=REGIME, n_components=N_COMPONENTS, bin_factor=1):
    """Fit the band ensemble and its Triage exactly as the paper does, for one configuration."""
    S = SCALERS[scale]
    prep = lambda X: S(binf(X, bin_factor))  # noqa: E731
    ax = binf(AXIS, bin_factor)
    ix = sp["ix"]
    if mode == "equal":
        bands = BandSet.equal_width(ax, n_bands, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    else:
        bands = BandSet.peak_informed(ax, binf(MM_all[ix["fit"]], bin_factor), n_bands,
                                      lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    lim = (prep(MM_all[ix["lim"]]), y_all[ix["lim"]])
    t0 = time.perf_counter()
    model = BandNegotiationClassifier(
        bands=bands, base_estimator=PLSDA(n_components=n_components), topology=topology,
        regime=regime, calibration=CALIBRATION, calibration_data=lim,
        calibration_split=CAL_SPLIT, random_state=SEED).fit(prep(MM_all[ix["fit"]]), y_all[ix["fit"]])
    t_fit = time.perf_counter() - t0
    tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,
                 n_components=n_components).fit(
        model, *lim, orientation_data=(prep(sp["sel_def"].X), sp["sel_def"].degraded))
    return {"model": model, "triage": tri, "bands": bands, "prep": prep, "t_fit": t_fit}


def localise(sp, fm, planted_intervals=None, trials=None):
    """Plant a local defect in a batch of 100 spectra of one qc isolate; does the chart name it?

    A hit is a named band that overlaps the planted interval; chance is the share of bands
    that overlap it. By default the seven published intervals, six trials each (42 trials).
    """
    prep, bands, model = fm["prep"], fm["bands"], fm["model"]
    edges, labs = [(b.lo, b.hi) for b in bands.bands], bands.labels
    rngb = np.random.default_rng(SEED + 11)
    ref = [prep(MM_all[np.sort(rngb.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))])
           for g in sp["chart_patients"]]
    prof = ControlProfile.fit(model, batches=ref, k=K_SIGMA, laser_nm=LASER_NM,
                              statistics=PROFILE_STATS, random_state=SEED)
    if trials is None:
        trials = [(lo, hi, np.random.default_rng(500 * t + rep))
                  for t, (lo, hi) in enumerate(REF_EDGES) for rep in range(LOCALISE_REPEATS)]
    hits, chance = [], []
    for lo, hi, r in trials:
        sel = (AXIS >= lo) & (AXIS < hi)
        overlap = [j for j, (a, b) in enumerate(edges) if a < hi and b > lo]
        g = r.choice(sp["p_qc"])
        Xl = MM_all[np.sort(r.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))].copy()
        Xl[:, sel] = Xl[:, sel] * 0.55 + r.normal(0, 0.02, (len(Xl), int(sel.sum())))
        tab = prof.check(prep(Xl), name="trial").table
        named = tab.assign(az=tab["z"].abs()).groupby("label")["az"].max().idxmax()
        hits.append(labs.index(named) in overlap)
        chance.append(len(overlap) / len(edges))
    return float(np.mean(hits)), float(np.mean(chance))


def evaluate(sp, *, loc=True, **config):
    """Every number the guide reports for one configuration on one split."""
    cfg = {"n_bands": N_BANDS, "topology": TOPOLOGY, "scale": SCALE_NAME, "mode": "equal",
           "regime": REGIME, "n_components": N_COMPONENTS, "bin_factor": 1, **config}
    fm = fit_model(sp, **cfg)
    ix, qd, prep = sp["ix"], sp["qc_def"], fm["prep"]
    y_qc = y_all[ix["qc"]]
    t0 = time.perf_counter()
    T = fm["triage"].assess(prep(qd.X), y_qc).table
    t_audit = time.perf_counter() - t0
    c = T["conflict"].to_numpy()
    fam = {k: roc_auc_score(qd.degraded[(qd.kind == k) | (qd.degraded == 0)],
                            c[(qd.kind == k) | (qd.degraded == 0)]) for k in degrade.PAPER_FAMILIES}
    row = {"repeat": sp["repeat"], **cfg,
           "channels_per_band": int(np.median([b.n_features for b in fm["bands"].bands])),
           "conflict_vs_degraded": roc_auc_score(qd.degraded, c),
           "conflict_vs_wrong": roc_auc_score(1 - T["correct"], c),
           "local": fam["local contamination"],
           "global_mean": float(np.mean([fam[k] for k in degrade.PAPER_FAMILIES if k != "local contamination"])),
           "acc_in_cohort": accuracy_score(y_all[ix["ref"]], fm["model"].predict(prep(MM_all[ix["ref"]]))),
           "acc_external": accuracy_score(y_B, fm["model"].predict(prep(MM_B))),
           "t_fit_s": fm["t_fit"], "t_audit_s": t_audit,
           "orientation_margin": fm["triage"].orientation_margin_}
    if loc:
        row["loc_hit"], row["loc_chance"] = localise(sp, fm)
    return row


In [2]:
def check(rows):
    """Compare this run with the values quoted in the manuscript and the response letter."""
    t = pd.DataFrame(rows, columns=["quantity", "this_run", "quoted", "tol"])
    t["ok"] = (t["this_run"] - t["quoted"]).abs() <= t["tol"]
    print(t.round(4).to_string(index=False))
    print("\nALL QUOTED VALUES REPRODUCE" if t["ok"].all() else
          f"\n{(~t['ok']).sum()} VALUES DO NOT REPRODUCE: correct the text or the run")
    return t

## 1 · The configurations

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def l2(X):
    X = np.asarray(X, float)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.where(n > 0, n, 1.0)


PREP = {"L2 + standardization": l2, "SNV": snv}   # per-channel standardization lives in the LR pipeline
ESTIMATOR = {
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "PLS-DA": lambda: PLSDA(n_components=N_COMPONENTS),
}
STEPS = [  # cumulative: each row changes one component of the row above
    ("[6]",                        dict(prep="L2 + standardization", est="logistic regression", cal=None,   topo="chain",    desc="eredg")),
    ("+ DG descriptor",            dict(prep="L2 + standardization", est="logistic regression", cal=None,   topo="chain",    desc="dg")),
    ("+ complete topology",        dict(prep="L2 + standardization", est="logistic regression", cal=None,   topo="complete", desc="dg")),
    ("+ SNV",                      dict(prep="SNV",                  est="logistic regression", cal=None,   topo="complete", desc="dg")),
    ("+ PLS-DA, calibrated (pypasi default)", dict(prep="SNV",       est="PLS-DA",              cal="auto", topo="complete", desc="dg")),
]


def run(sp, prep, est, cal, topo, desc):
    P = PREP[prep]
    ix = sp["ix"]
    lim = (P(MM_all[ix["lim"]]), y_all[ix["lim"]])
    model = BandNegotiationClassifier(
        bands=REF_BANDS, base_estimator=ESTIMATOR[est](), topology=topo, regime=REGIME,
        calibration=cal, calibration_data=lim if cal else None, random_state=SEED,
    ).fit(P(MM_all[ix["fit"]]), y_all[ix["fit"]])
    tri = Triage(conflict_score=desc, review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,
                 n_components=N_COMPONENTS).fit(
        model, *lim, orientation_data=(P(sp["sel_def"].X), sp["sel_def"].degraded))
    y_qc, qd = y_all[ix["qc"]], sp["qc_def"]
    Td = tri.assess(P(qd.X), y_qc).table
    T18 = tri.assess(P(MM_all[ix["qc"]]), y_qc).table
    T19 = tri.assess(P(MM_B), y_B).table
    lab = np.r_[np.zeros(len(T18)), np.ones(len(T19))]
    return {"planted defects": roc_auc_score(qd.degraded, Td["conflict"]),
            "real change": roc_auc_score(lab, np.r_[T18["conflict"], T19["conflict"]]),
            "external accuracy": accuracy_score(y_B, model.predict(P(MM_B)))}

## 2 · Five splits

In [4]:
rows = []
for rep in range(5):
    sp = make_split(rep)
    for name, cfg in STEPS:
        rows.append({"split": rep, "configuration": name, **run(sp, **cfg)})
    print(f"split {rep} done", flush=True)
res = pd.DataFrame(rows)
order = [n for n, _ in STEPS]
g = res.drop(columns="split").groupby("configuration").agg(["mean", "std"]).reindex(order)
table = pd.DataFrame({k: g[(k, "mean")].round(3).astype(str) + " ± " + g[(k, "std")].round(3).astype(str)
                      for k in ("planted defects", "real change", "external accuracy")})
print(table.to_string())

base, final = res[res.configuration == "[6]"].set_index("split"), \
    res[res.configuration == order[-1]].set_index("split")
for k in ("planted defects", "real change"):
    d = final[k] - base[k]
    print(f"\n{k}: pypasi default minus [6] = {d.mean():+.3f} ± {d.std():.3f}, better in {(d > 0).sum()}/5 splits")

/tmp/ipykernel_591/2548038664.py:34: WeakOrientationWarning: the orientation of 'eredg' was decided on an AUROC of 0.482 against the supplied orientation data - a margin of 0.018 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score=desc, review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


/tmp/ipykernel_591/2548038664.py:34: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.518 against the supplied orientation data - a margin of 0.018 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score=desc, review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


/tmp/ipykernel_591/2548038664.py:34: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.544 against the supplied orientation data - a margin of 0.044 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score=desc, review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


split 0 done


/tmp/ipykernel_591/2548038664.py:34: WeakOrientationWarning: the orientation of 'eredg' was decided on an AUROC of 0.456 against the supplied orientation data - a margin of 0.044 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score=desc, review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


/tmp/ipykernel_591/2548038664.py:34: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.549 against the supplied orientation data - a margin of 0.049 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score=desc, review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


... (7 outputs left out)

split 3 done


/tmp/ipykernel_591/2548038664.py:34: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.525 against the supplied orientation data - a margin of 0.025 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score=desc, review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


split 4 done


                                      planted defects    real change external accuracy
configuration                                                                         
[6]                                     0.539 ± 0.015  0.603 ± 0.015      0.78 ± 0.021
+ DG descriptor                          0.52 ± 0.019    0.57 ± 0.03      0.78 ± 0.021
+ complete topology                      0.551 ± 0.01  0.612 ± 0.043     0.792 ± 0.025
+ SNV                                   0.626 ± 0.015  0.575 ± 0.048      0.802 ± 0.03
+ PLS-DA, calibrated (pypasi default)    0.76 ± 0.028  0.749 ± 0.053     0.771 ± 0.026

planted defects: pypasi default minus [6] = +0.220 ± 0.032, better in 5/5 splits

real change: pypasi default minus [6] = +0.146 ± 0.040, better in 5/5 splits


## 3 · Every value quoted, checked

In [5]:
M = res.groupby("configuration").mean(numeric_only=True)
S6, SD = "[6]", order[-1]
rows = []
quoted = {  # planted defects, real change, external accuracy (means over five splits)
    "[6]":                                   (0.539, 0.603, 0.780),
    "+ DG descriptor":                       (0.520, 0.570, 0.780),
    "+ complete topology":                   (0.551, 0.612, 0.792),
    "+ SNV":                                 (0.626, 0.575, 0.802),
    "+ PLS-DA, calibrated (pypasi default)": (0.760, 0.749, 0.771),
}
for name, (a, b, e) in quoted.items():
    rows += [(f"{name}: planted defects", M.loc[name, "planted defects"], a, 0.005),
             (f"{name}: real change", M.loc[name, "real change"], b, 0.005),
             (f"{name}: external accuracy", M.loc[name, "external accuracy"], e, 0.005)]
dp = res[res.configuration == SD].set_index("split")["planted defects"] - res[res.configuration == S6].set_index("split")["planted defects"]
dr = res[res.configuration == SD].set_index("split")["real change"] - res[res.configuration == S6].set_index("split")["real change"]
rows += [("default minus [6], planted defects", dp.mean(), 0.220, 0.005),
         ("default minus [6], real change", dr.mean(), 0.146, 0.005),
         ("better than [6] in (of 5), planted", float((dp > 0).sum()), 5, 0),
         ("better than [6] in (of 5), real change", float((dr > 0).sum()), 5, 0)]
claims = check(rows)
res.to_csv("method_paper_comparison.csv", index=False)

                                                quantity  this_run  quoted   tol   ok
                                    [6]: planted defects    0.5393   0.539 0.005 True
                                        [6]: real change    0.6029   0.603 0.005 True
                                  [6]: external accuracy    0.7803   0.780 0.005 True
                        + DG descriptor: planted defects    0.5198   0.520 0.005 True
                            + DG descriptor: real change    0.5704   0.570 0.005 True
                      + DG descriptor: external accuracy    0.7803   0.780 0.005 True
                    + complete topology: planted defects    0.5506   0.551 0.005 True
                        + complete topology: real change    0.6125   0.612 0.005 True
                  + complete topology: external accuracy    0.7922   0.792 0.005 True
                                  + SNV: planted defects    0.6265   0.626 0.005 True
                                      + SNV: real chan